In [23]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.feature_selection import SelectKBest, f_regression

df = pd.read_csv("github_repo_features.csv")

df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)
ref_date = datetime(2025, 5, 1)

df["project_age"] = (ref_date - df["created_at"]).dt.days
df["days_since_update"] = (ref_date - df["updated_at"]).dt.days
df["days_since_push"] = (ref_date - df["pushed_at"]).dt.days

df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
df["update_rate"] = 1 / (1 + df["days_since_update"])

#New features based on correlation matrix
df["stars_per_fork"] = df["stars"] / (df["forks"] + 1)
df["stars_per_commit"] = df["stars"] / (df["commits_count"] + 1)
df["activity_level"] = df["commits_count"] + df["open_issues"] + df["subscribers_count"]
df["recent_activity"] = df["update_rate"] * df["commits_count"]
df["forks_times_commits"] = df["forks"] * df["commits_count"]
df["social_popularity"] = df["forks"] + df["watchers"]

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

for col in ["has_wiki", "has_projects", "has_downloads", "is_fork", "archived"]:
    df[col] = df[col].astype(int)

features = [
    'open_issues', 'size', 'has_wiki', 'has_projects', 'has_downloads',
    'is_fork', 'archived', 'language', 'license', 'subscribers_count',
    'contributors_count', 'commits_count', 'readme_size', 'project_age',
    'days_since_update', 'days_since_push', 'forks_per_day', 'update_rate',
    'stars_per_fork', 'stars_per_commit', 'activity_level', 'recent_activity',
    'forks_times_commits', 'social_popularity'
]

X = df[features]
y = df["stars"]

#preprocessing
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_regression, k=15)),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

#spliting  and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

#Evaluate
predictions = pipeline.predict(X_test)
print("R2 score on test set:", r2_score(y_test, predictions))


R2 score on test set: 0.994256638812807
